# Imports

All packages and project modules used by this notebook.


In [ ]:
import os
import sys
import random

import torch
import numpy as np
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import random_split

from src.config import (
    DEVICE,
    BATCH_SIZE,
    EPOCHS_FINETUNE,
    DATASET_NYU_PATH,
    DATASET_KITTI_PATH,
    MODEL_SAVE_PATH,
    JSON_DIR,
    SSL_CKPT_PATH_KITTI,
    SSL_CKPT_PATH_NYU,
)
from src.dataset import (
    DepthEstimationDataset,
    build_dataloader,
    random_subset,
    scene_level_split,
)
from src.train import LeJEPAMeter, METERFinetune, get_callbacks
from src.utils import save_dict_to_json


# Globals

Notebook-wide variables, reproducibility settings, and environment setup.


In [ ]:
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
SEED = 42
DATASET = "kitti"  # Change to "nyu" if you want to test on NYU instead of KITTI
MAX_SAMPLES = 70000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
L.seed_everything(SEED, workers=True)

generator = torch.Generator().manual_seed(SEED)

if torch.cuda.is_available() and torch.cuda.get_device_name(0) == "Tesla T4":
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, "/content/drive/MyDrive/CV-project-25-26")

print(f"[OK] Environment ready on {DEVICE}")
print(f"[INFO] Dataset: {DATASET} | max samples: {MAX_SAMPLES}")


# Data

Dataset construction, subsets, scene-level splits, and dataloaders.


In [ ]:
print("\n=== SSL PRETRAIN DATALOADER CHECK ===")


# ==========================================
# 1. KITTI FLOW (official pre-split)
# ==========================================
if DATASET == "kitti":
    # --- SSL Pretrain ---
    ssl_dataset_train = DepthEstimationDataset(
        dataset="kitti", path=DATASET_KITTI_PATH, mode="pretrain", depth=False, 
        split="train"
    )
    subset_size_ssl = min(MAX_SAMPLES, len(ssl_dataset_train))
    ssl_dataset_train = random_subset(ssl_dataset_train, subset_size_ssl)
    
    # For KITTI SSL, create a small validation set by splitting the subset 80/20
    train_size_ssl = int(0.8 * len(ssl_dataset_train))
    val_size_ssl = len(ssl_dataset_train) - train_size_ssl
    ssl_dataset_train, ssl_dataset_val = random_split(ssl_dataset_train, [train_size_ssl, val_size_ssl], generator=generator)
    
    ssl_dataset_test = DepthEstimationDataset(
        dataset="kitti", path=DATASET_KITTI_PATH, mode="pretrain", depth=False,
        split="test"
    )

    # --- Supervised Fine-Tuning ---
    ft_dataset_train_full = DepthEstimationDataset(
        dataset="kitti", path=DATASET_KITTI_PATH, mode="finetune", depth=True,
        split="train"
    )
    subset_size_ft = min(MAX_SAMPLES, len(ft_dataset_train_full))
    ft_dataset_train_subset = random_subset(ft_dataset_train_full, subset_size_ft)
    
    # Run a sequence-level split for KITTI fine-tuning
    ft_dataset_train, ft_dataset_val = scene_level_split(ft_dataset_train_subset, train_ratio=0.8, seed=SEED)

    ft_dataset_test = DepthEstimationDataset(
        dataset="kitti", path=DATASET_KITTI_PATH, mode="finetune", depth=True,
        split="test"
    )
    

# ==========================================
# 2. NYU FLOW (dynamic split from .mat)
# ==========================================
else:
    # --- SSL Pretrain ---
    ssl_dataset_full = DepthEstimationDataset(
        dataset="nyu", path=DATASET_NYU_PATH, mode="pretrain", depth=False, split="train"
    )
    subset_size_ssl = min(MAX_SAMPLES, len(ssl_dataset_full))
    ssl_dataset_full = random_subset(ssl_dataset_full, subset_size_ssl)

    train_size_ssl = int(0.8 * len(ssl_dataset_full))
    val_size_ssl = len(ssl_dataset_full) - train_size_ssl 
    ssl_dataset_train, ssl_dataset_val = random_split(
        ssl_dataset_full, [train_size_ssl, val_size_ssl], generator=generator
    )

    ssl_dataset_test = DepthEstimationDataset(
        dataset="nyu", path=DATASET_NYU_PATH, mode="pretrain", depth=False, split="test"
    )

    # --- Supervised Fine-Tuning ---
    ft_dataset_full = DepthEstimationDataset(
        dataset="nyu", path=DATASET_NYU_PATH, mode="finetune", depth=True, split="train"
    )
    subset_size_ft = min(MAX_SAMPLES, len(ft_dataset_full))
    ft_dataset_full = random_subset(ft_dataset_full, subset_size_ft)

    # Run a scene-level split for NYU fine-tuning
    ft_dataset_train, ft_dataset_val = scene_level_split(ft_dataset_full, train_ratio=0.8, seed=SEED)

    ft_dataset_test = DepthEstimationDataset(
        dataset="nyu", path=DATASET_NYU_PATH, mode="finetune", depth=True, split="test"
    )

print(f"\nDataset      : {DATASET.upper()}")
print(f"Pretrain Train samples: {len(ssl_dataset_train)}")
print(f"Pretrain Val samples  : {len(ssl_dataset_val)}")
print(f"Pretrain Test samples : {len(ssl_dataset_test)}")
print(f"FT Train samples: {len(ft_dataset_train)}")
print(f"FT Val samples  : {len(ft_dataset_val)}")
print(f"FT Test samples : {len(ft_dataset_test)}")
print("Datasets prepared successfully")


# Train

Training cycle orchestration for SSL pretraining and supervised/fine-tuning runs.


## LeJEPA SSL Pretraining Setup


In [ ]:
# DataLoaders
ssl_loader_train = build_dataloader(
    ssl_dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)
ssl_loader_val = build_dataloader(
    ssl_dataset_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)


print("\n=== EJEPA SSL PRETRAINING ===")

accelerator = "gpu" if torch.cuda.is_available() or torch.backends.mps.is_available() else "cpu"
precision = "bf16-mixed" if torch.cuda.is_available() or torch.backends.mps.is_available() else "32-true"
print(f"Accelerator : {accelerator}")
print(f"Precision   : {precision}")

ssl_model = LeJEPAMeter(
    arch_type="xxs",
    lamb=0.02, 
    lr=5e-4
)

trainer = L.Trainer(
    max_epochs=50,
    accelerator=accelerator,
    devices=1,
    precision=precision,
    callbacks=get_callbacks(MODEL_SAVE_PATH.joinpath("pretrain"), dataset_name=DATASET, arch_name=ssl_model.arch_type),
    logger=CSVLogger("logs", name=f"lejepa_pretrain_{DATASET}_{ssl_model.arch_type}"),
)
print(f"Accelerator: {accelerator} | precision: {precision}")

trainer.fit(
    ssl_model, 
    train_dataloaders=ssl_loader_train,
    val_dataloaders=ssl_loader_val
)
print("[OK] SSL pretraining completed")

## Fine-Tuning with LeJEPA Pretraining


In [ ]:
print("\n=== SUPERVISED FINE-TUNING CONFIG ===")

# DataLoaders
ft_loader_train = build_dataloader(ft_dataset_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
ft_loader_val = build_dataloader(ft_dataset_val, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
ft_loader_test = build_dataloader(ft_dataset_test, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

# METERFinetune Model Initialization
ft_model = METERFinetune(
    arch_type="xxs",
    lr_enc=1e-5,   
    lr_dec=1e-3,   
    pretrained_ckpt=SSL_CKPT_PATH_KITTI if DATASET == "kitti" else SSL_CKPT_PATH_NYU, 
    freeze_encoder=False,
    use_step_lr=True,
    dataset=DATASET
)

# Trainer Configuration
accelerator = "gpu" if torch.cuda.is_available() or torch.backends.mps.is_available() else "cpu"
precision = "bf16-mixed" if torch.cuda.is_available() or torch.backends.mps.is_available() else "32-true"

ft_trainer = L.Trainer(
    max_epochs=60, 
    accumulate_grad_batches=4,
    gradient_clip_val=1.0,
    accelerator=accelerator,
    devices=1,
    precision=precision,
    callbacks=get_callbacks(MODEL_SAVE_PATH.joinpath("finetune"), monitor="val/delta1", dataset_name=DATASET, arch_name=ft_model.arch_type, monitor_mode="max"),
    logger=CSVLogger("logs", name=f"meter_finetune_{DATASET}_{ft_model.arch_type}"),
    enable_model_summary=False
)

# Start Fine-Tuning
print(f"Accelerator: {accelerator} | Precision: {precision}")
print("[INFO] Starting Fine-Tuning with real depth labels...")

ft_trainer.fit(ft_model, train_dataloaders=ft_loader_train,val_dataloaders=ft_loader_val)
print("\n[INFO] Running final validation...")

lejepa_results = ft_trainer.validate(ft_model, dataloaders=ft_loader_val,verbose=False)[0]

metrics_data = {
    "RMSE": float(lejepa_results.get("val/rmse", 0)),
    "AbsRel": float(lejepa_results.get("val/abs_rel", 0)),
    "Delta1": float(lejepa_results.get("val/delta1", 0)),
    "L1 Depth Error": float(lejepa_results.get("val/l1", 0)),
    "train_time_sec": getattr(ft_model, "total_train_time", 0.0)
}

print("\n=== FINAL VALIDATION RESULTS ===")
print(metrics_data)

save_path = JSON_DIR / f"lejepa_metrics_{DATASET}_{ft_model.arch_type}.json"
save_dict_to_json(metrics_data, save_path)

print(f"[OK] Metrics saved to: {save_path}")

## Supervised Training from Scratch


In [ ]:
print("\n=== SUPERVISED FINE-TUNING CONFIG ===")

# DataLoaders 
ft_loader_train = build_dataloader(ft_dataset_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
ft_loader_val = build_dataloader(ft_dataset_val, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
ft_loader_test = build_dataloader(ft_dataset_test, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

# METERFinetune Model Initialization


ft_model = METERFinetune(
    arch_type="xxs",
    lr_enc=1e-3,   
    lr_dec=1e-3,   
    freeze_encoder=False,
    warmup_epochs=3,
    dataset=DATASET
)

# Trainer Configuration
accelerator = "gpu" if torch.cuda.is_available() or torch.backends.mps.is_available() else "cpu"
precision = "bf16-mixed" if torch.cuda.is_available() or torch.backends.mps.is_available() else "32-true"

ft_trainer = L.Trainer(
    max_epochs=EPOCHS_FINETUNE, 
    accumulate_grad_batches=4,
    accelerator=accelerator,
    gradient_clip_val=1.0,
    devices=1,
    precision=precision,
    callbacks=get_callbacks(MODEL_SAVE_PATH.joinpath("supervised"), monitor="val/delta1", dataset_name=DATASET, arch_name=ft_model.arch_type, monitor_mode="max"),
    logger=CSVLogger("logs", name=f"meter_sl_{DATASET}_{ft_model.arch_type}"),
    enable_model_summary=False,
)

# Start Fine-Tuning
print(f"Accelerator: {accelerator} | Precision: {precision}")
print("[INFO] Starting SL with real depth labels...")

ft_trainer.fit(ft_model, train_dataloaders=ft_loader_train,val_dataloaders=ft_loader_val)
print("\n[INFO] Running final validation...")

lejepa_results = ft_trainer.validate(ft_model, dataloaders=ft_loader_val,verbose=False)[0]

metrics_data = {
    "RMSE": float(lejepa_results.get("val/rmse", 0)),
    "AbsRel": float(lejepa_results.get("val/abs_rel", 0)),
    "Delta1": float(lejepa_results.get("val/delta1", 0)),
    "L1 Depth Error": float(lejepa_results.get("val/l1", 0)),
    "train_time_sec": getattr(ft_model, "total_train_time", 0.0)
}

print("\n=== FINAL VALIDATION RESULTS ===")
print(metrics_data)

save_path = JSON_DIR / f"sl_metrics_{DATASET}_{ft_model.arch_type}.json"
save_dict_to_json(metrics_data, save_path)

print(f"[OK] Metrics saved to: {save_path}")